# 01 · EDA: Breakfast at the Frat

Sanity checks on the weekly product panel. All logic lives in `src/`; this notebook only calls it and shows the results.

Prerequisite: `python -m src.pipeline prep` (writes `data/processed/panel.parquet`).

In [ ]:
# reload edited src/ modules automatically (avoids stale imports after code changes)
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

from src.config import FIGURES, PANEL_PATH
from src.plots import EdaPlotter, FigureStore, PlotStyle
from src.schema import Col

style = PlotStyle()
style.apply()
plotter = EdaPlotter(style)
store = FigureStore(FIGURES)
panel = pd.read_parquet(PANEL_PATH)
panel.shape

## Coverage
One row per product-week. Gaps are weeks with no recorded sales.

In [ ]:
panel.groupby(Col.CATEGORY).agg(
    products=(Col.UPC, "nunique"),
    weeks=(Col.WEEK, "nunique"),
    units=(Col.UNITS, "sum"),
    eligible=(Col.ELIGIBLE, "mean"),
)

In [ ]:
store.save(plotter.coverage_heatmap(panel), "01_coverage")

## Filters
Products that fail `min_weeks` or `min_price_variation` keep their rows but get an imputed elasticity in the elasticity step (notebook 02).

In [ ]:
products = panel.drop_duplicates(Col.UPC)
products.groupby([Col.CATEGORY, Col.FILTER_REASON]).size().unstack(fill_value=0)

## Price vs units
Within-product (demeaned) log-log scatter. The slope is a **naive** elasticity: there are no promo, seasonality or trend controls yet. The elasticity models (notebook 02) add them.

In [ ]:
store.save(plotter.price_vs_units(panel), "01_price_vs_units")

## Promo activity
How often each tactic runs. This sets how much signal the elasticity models have for promo lift.

In [ ]:
store.save(plotter.promo_frequency(panel), "01_promo_frequency")

## LOESS trend
The trend is fitted on training weeks only and held flat over the shaded holdout.

In [ ]:
top = panel.groupby(Col.UPC)[Col.UNITS].sum().nlargest(4).index.tolist()
store.save(plotter.trend_overlay(panel, top), "01_trend")